# Cascade S2S Demo: Listen to Translated Audio

Runs the full cascaded speech-to-speech pipeline on a handful of audio samples
and renders the source + translation + target audio inline so you can listen.

**Pipeline:** `source audio  ->  Whisper ASR  ->  NLLB MT  ->  MMS-TTS  ->  target audio`

**Use cases:**
1. Sanity-check the cascade quality before training the Hibiki S2ST model
2. Produce demo audio for the FYP report appendix
3. Compare vanilla Whisper-small vs. the pseudo-label fine-tuned variant

Both directions are supported: `Sw -> En` (default) and `En -> Sw`.

**Outputs** written to `/kaggle/working/cascade_demo/<direction>/`:
- `source/sample_NN.wav` — original source audio
- `target/sample_NN_<asr_variant>.wav` — synthesised target audio
- `transcripts.jsonl` — one JSON line per sample with all texts
- `index.md` — side-by-side markdown table of texts + audio links

In [ ]:
!pip install -q transformers accelerate torchaudio soundfile sentencepiece faster-whisper "datasets<4.0"

In [ ]:
# HuggingFace login (FLEURS download)
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HF login OK')
except Exception as e:
    print(f'HF token not set ({e}); FLEURS may rate-limit')

In [ ]:
import os, sys, json, gc, shutil
from pathlib import Path
import numpy as np
import torch
import soundfile as sf

REPO_DIR = '/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw'
sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

# === CONFIGURATION ===
DIRECTION   = 'sw2en'        # 'sw2en' or 'en2sw'
NUM_SAMPLES = 5              # how many to demo

# Root where the four fine-tuned Whisper variants live (each as <name>/final/).
# This is the standard layout produced by whisper_asr/run.sh.
# On Kaggle: upload the runs directory as a dataset and point this here.
# On AWS A10G: just point at /home/ec2-user/data/asr_runs (the run.sh default).
WHISPER_FT_ROOT = '/kaggle/input/datasets/victormugambi/whisper-ft'

# ASR variants to compare — listed in worst-to-best order from Table 11.
# The vanilla baseline is included so the demo shows the *delta* from fine-tuning.
# Comment out any variant you have not uploaded.
ASR_VARIANTS = [
    ('vanilla_small',                 'openai/whisper-small'),
    ('ft_kenspeech_only',             f'{WHISPER_FT_ROOT}/ft_kenspeech_only/final'),
    ('ft_kenspeech_pseudo_raw',       f'{WHISPER_FT_ROOT}/ft_kenspeech_pseudo_raw/final'),
    ('ft_kenspeech_pseudo_filtered',  f'{WHISPER_FT_ROOT}/ft_kenspeech_pseudo_filtered/final'),
    ('ft_kenspeech_gold_upper_bound', f'{WHISPER_FT_ROOT}/ft_kenspeech_gold_upper_bound/final'),
]
# Drop entries whose path doesn't exist (so a partial upload still runs)
ASR_VARIANTS = [
    (name, ckpt) for name, ckpt in ASR_VARIANTS
    if ckpt.startswith('openai/') or os.path.isdir(ckpt)
]

# Source audio. Default: FLEURS sw_ke test (Sw->En) or en_us test (En->Sw).
USE_FLEURS    = True
CUSTOM_WAV_DIR = None  # alternative: directory of WAVs to translate (any sample rate; resampled below)

# Working dirs
OUT_DIR = f'/kaggle/working/cascade_demo/{DIRECTION}'
os.makedirs(f'{OUT_DIR}/source', exist_ok=True)
os.makedirs(f'{OUT_DIR}/target', exist_ok=True)

# Direction-specific defaults
if DIRECTION == 'sw2en':
    SOURCE_LANG, TARGET_LANG = 'sw', 'en'
    SRC_NLLB, TGT_NLLB       = 'swh_Latn', 'eng_Latn'
    FLEURS_SUBSET            = 'sw_ke'
    MMS_TTS_LANG             = 'eng'
else:
    SOURCE_LANG, TARGET_LANG = 'en', 'sw'
    SRC_NLLB, TGT_NLLB       = 'eng_Latn', 'swh_Latn'
    FLEURS_SUBSET            = 'en_us'
    MMS_TTS_LANG             = 'swh'

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f'  GPU: {used:.1f}/{total:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Direction:    {DIRECTION}')
print(f'GPU:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'ASR variants: {[v[0] for v in ASR_VARIANTS]}  ({len(ASR_VARIANTS)} found)')
if not ASR_VARIANTS:
    raise RuntimeError(
        f'No ASR variants found. Either {WHISPER_FT_ROOT} is missing, '
        'or none of the variant subdirs exist. Set WHISPER_FT_ROOT and try again.'
    )
print(f'Output:       {OUT_DIR}')

## Step 1: Collect source audio samples

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torchaudio.functional as taf

# preds[variant_name][sample_idx] = transcript
asr_preds = {name: {} for name, _ in ASR_VARIANTS}

def load_asr(ckpt: str):
    """Load a Whisper checkpoint.

    Local fine-tuned dirs (output of whisper_asr/train.py) are HF transformers
    checkpoints and are loaded with WhisperForConditionalGeneration directly.
    Vanilla 'openai/...' ids try faster-whisper first (CTranslate2, faster on T4)
    and fall back to transformers if conversion is unavailable.
    """
    is_local = os.path.isdir(ckpt)
    if not is_local:
        try:
            from faster_whisper import WhisperModel
            wm = WhisperModel(
                ckpt,
                device='cuda' if torch.cuda.is_available() else 'cpu',
                compute_type='int8',
            )
            return ('faster_whisper', wm, None)
        except Exception as e:
            print(f'  faster-whisper unavailable for {ckpt} ({e}); using transformers')

    proc = WhisperProcessor.from_pretrained(ckpt)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = WhisperForConditionalGeneration.from_pretrained(ckpt, torch_dtype=dtype).to(device)
    model.eval()
    return ('transformers', model, proc)


def transcribe(backend: str, model, proc, wav: np.ndarray, sr: int) -> str:
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != 16000:
        wav = taf.resample(torch.from_numpy(wav.astype(np.float32)), sr, 16000).numpy()
    wav = wav.astype(np.float32)

    if backend == 'faster_whisper':
        segments, _ = model.transcribe(wav, language=SOURCE_LANG, task='transcribe', beam_size=1)
        return ''.join(seg.text for seg in segments).strip()

    inp = proc(wav, sampling_rate=16000, return_tensors='pt')
    feats = inp.input_features.to(device, dtype=next(model.parameters()).dtype)
    with torch.no_grad():
        ids = model.generate(
            feats,
            language=SOURCE_LANG, task='transcribe',
            num_beams=1, do_sample=False, max_new_tokens=225,
        )
    return proc.batch_decode(ids, skip_special_tokens=True)[0].strip()


for name, ckpt in ASR_VARIANTS:
    print(f'\n=== ASR: {name}  ({ckpt}) ===')
    backend, model, proc = load_asr(ckpt)
    print(f'  backend: {backend}')
    for s in samples:
        wav, sr = sf.read(s['source_wav'])
        text = transcribe(backend, model, proc, wav, sr)
        asr_preds[name][s['idx']] = text
        print(f'  [{s["idx"]}] {name}: {text[:100]}')
    # Free this variant before loading the next
    del model
    if proc is not None:
        del proc
    free_gpu()

## Step 2: ASR — transcribe source audio with each Whisper variant

Loads each ASR variant in turn (one at a time to fit on T4), runs greedy decoding with the source-language prompt, and stores predictions per variant.

In [ ]:
from faster_whisper import WhisperModel

# preds[variant_name][sample_idx] = transcript
asr_preds = {name: {} for name, _ in ASR_VARIANTS}

for name, ckpt in ASR_VARIANTS:
    print(f'\n=== ASR: {name}  ({ckpt}) ===')
    # faster-whisper accepts HF model IDs and local CTranslate2 dirs.
    # For HF transformers checkpoints (the fine-tuned student), set compute_type='int8' and
    # a converted CTranslate2 directory; or fall back to the transformers pipeline if needed.
    try:
        wm = WhisperModel(ckpt, device='cuda' if torch.cuda.is_available() else 'cpu', compute_type='int8')
        backend = 'faster_whisper'
    except Exception as e:
        print(f'  faster-whisper rejected {ckpt} ({e}); falling back to transformers')
        from transformers import WhisperProcessor, WhisperForConditionalGeneration
        proc  = WhisperProcessor.from_pretrained(ckpt)
        model = WhisperForConditionalGeneration.from_pretrained(ckpt).to(device)
        model.eval()
        backend = 'transformers'

    for s in samples:
        wav, sr = sf.read(s['source_wav'])
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        # Resample to 16kHz for Whisper
        if sr != 16000:
            import torchaudio.functional as taf
            wav_t = torch.from_numpy(wav.astype(np.float32))
            wav = taf.resample(wav_t, sr, 16000).numpy()
            sr = 16000

        if backend == 'faster_whisper':
            segments, _info = wm.transcribe(wav.astype(np.float32), language=SOURCE_LANG, task='transcribe', beam_size=1)
            text = ''.join(seg.text for seg in segments).strip()
        else:
            inp = proc(wav, sampling_rate=16000, return_tensors='pt').to(device)
            with torch.no_grad():
                ids = model.generate(
                    inp.input_features,
                    language=SOURCE_LANG, task='transcribe',
                    num_beams=1, do_sample=False, max_new_tokens=225,
                )
            text = proc.batch_decode(ids, skip_special_tokens=True)[0].strip()

        asr_preds[name][s['idx']] = text
        print(f'  [{s["idx"]}] {name}: {text[:100]}')

    # Free this variant before loading the next
    if backend == 'faster_whisper':
        del wm
    else:
        del model, proc
    free_gpu()

## Step 3: MT — translate each transcript with NLLB-200

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_MODEL = 'facebook/nllb-200-distilled-1.3B'
print(f'Loading {MT_MODEL}...')
mt_tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang=SRC_NLLB)
mt_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, dtype=mt_dtype).to(device)
mt_model.eval()
tgt_id = mt_tok.convert_tokens_to_ids(TGT_NLLB)
free_gpu()

# mt_preds[variant][idx] = en_text
mt_preds = {name: {} for name, _ in ASR_VARIANTS}
for name, _ in ASR_VARIANTS:
    print(f'\n=== MT for ASR variant: {name} ===')
    for s in samples:
        sw = asr_preds[name][s['idx']] or ' '
        enc = mt_tok(sw, return_tensors='pt', truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = mt_model.generate(
                **enc,
                forced_bos_token_id=tgt_id,
                num_beams=4, max_new_tokens=256,
            )
        en = mt_tok.batch_decode(out, skip_special_tokens=True)[0].strip()
        mt_preds[name][s['idx']] = en
        print(f'  [{s["idx"]}] {sw[:60]}  ->  {en[:80]}')

del mt_model, mt_tok
free_gpu()

## Step 4: TTS — synthesise target audio with MMS-TTS

In [ ]:
from transformers import VitsModel, VitsTokenizer

TTS_NAME = f'facebook/mms-tts-{MMS_TTS_LANG}'
print(f'Loading {TTS_NAME}...')
tts_tok = VitsTokenizer.from_pretrained(TTS_NAME)
tts_model = VitsModel.from_pretrained(TTS_NAME).to(device)
tts_model.eval()
tts_sr = tts_model.config.sampling_rate
free_gpu()

# target_wavs[variant][idx] = path
target_wavs = {name: {} for name, _ in ASR_VARIANTS}
for name, _ in ASR_VARIANTS:
    print(f'\n=== TTS for ASR variant: {name} ===')
    for s in samples:
        en = mt_preds[name][s['idx']] or ' '
        inputs = tts_tok(en, return_tensors='pt').to(device)
        with torch.no_grad():
            out = tts_model(**inputs).waveform
        wav = out.squeeze().cpu().float().numpy()
        # Peak normalise to avoid clipping
        peak = float(np.max(np.abs(wav)))
        if peak > 0:
            wav = wav * (0.95 / peak)
        out_path = f'{OUT_DIR}/target/sample_{s["idx"]:02d}_{name}.wav'
        sf.write(out_path, wav.astype(np.float32), tts_sr)
        target_wavs[name][s['idx']] = out_path
        dur = len(wav) / tts_sr
        print(f'  [{s["idx"]}] {dur:.1f}s -> {out_path}')

del tts_model, tts_tok
free_gpu()

## Step 5: Render side-by-side for listening

In [ ]:
import IPython.display as ipd
from IPython.display import display, HTML, Markdown

for s in samples:
    display(HTML(f"<h3>Sample {s['idx']:02d}</h3>"))
    if s.get('source_text_gold'):
        display(Markdown(f"**Source ({SOURCE_LANG}) gold transcript:** {s['source_text_gold']}"))
    display(Markdown(f"**Source audio ({SOURCE_LANG}):**"))
    display(ipd.Audio(s['source_wav']))

    for name, _ in ASR_VARIANTS:
        display(Markdown(
            f"**[{name}] ASR:** {asr_preds[name][s['idx']]}  \n"
            f"**[{name}] MT  ->  {TARGET_LANG}:** {mt_preds[name][s['idx']]}"
        ))
        display(Markdown(f"**[{name}] TTS output ({TARGET_LANG}):**"))
        display(ipd.Audio(target_wavs[name][s['idx']]))

    if s.get('reference_target'):
        display(Markdown(f"**Reference target text ({TARGET_LANG}):** {s['reference_target']}"))
    display(HTML('<hr>'))

## Step 6: Persist transcripts + index

Writes a `transcripts.jsonl` (machine-readable) and an `index.md` (human-readable) so the demo can be re-opened later or inserted into the FYP report appendix.

In [ ]:
# transcripts.jsonl
with open(f'{OUT_DIR}/transcripts.jsonl', 'w', encoding='utf-8') as f:
    for s in samples:
        entry = {
            'idx': s['idx'],
            'fleurs_id': s.get('fleurs_id'),
            'source_wav': os.path.relpath(s['source_wav'], OUT_DIR),
            'source_text_gold': s.get('source_text_gold'),
            'reference_target': s.get('reference_target'),
            'asr': {name: asr_preds[name][s['idx']] for name, _ in ASR_VARIANTS},
            'mt':  {name: mt_preds[name][s['idx']]  for name, _ in ASR_VARIANTS},
            'target_wavs': {name: os.path.relpath(target_wavs[name][s['idx']], OUT_DIR) for name, _ in ASR_VARIANTS},
        }
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')
print(f'Wrote {OUT_DIR}/transcripts.jsonl')

# index.md
lines = [f'# Cascade S2S Demo — {DIRECTION}', '']
for s in samples:
    lines.append(f'## Sample {s["idx"]:02d}  (fleurs_id={s.get("fleurs_id")})')
    if s.get('source_text_gold'):
        lines.append(f'- **Source gold:** {s["source_text_gold"]}')
    lines.append(f'- **Source audio:** [`{os.path.basename(s["source_wav"])}`](source/{os.path.basename(s["source_wav"])})')
    for name, _ in ASR_VARIANTS:
        lines.append(f'- **[{name}] ASR:** {asr_preds[name][s["idx"]]}')
        lines.append(f'- **[{name}] MT:** {mt_preds[name][s["idx"]]}')
        tgt_rel = os.path.basename(target_wavs[name][s["idx"]])
        lines.append(f'- **[{name}] TTS:** [`{tgt_rel}`](target/{tgt_rel})')
    if s.get('reference_target'):
        lines.append(f'- **Target reference:** {s["reference_target"]}')
    lines.append('')
Path(f'{OUT_DIR}/index.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'Wrote {OUT_DIR}/index.md')

# Zip for download
zip_base = f'/kaggle/working/cascade_demo_{DIRECTION}'
shutil.make_archive(zip_base, 'zip', root_dir=OUT_DIR)
print(f'Zipped: {zip_base}.zip')

In [ ]:
# Quick summary
for s in samples:
    print(f'\n[{s["idx"]:02d}] {SOURCE_LANG.upper()} -> {TARGET_LANG.upper()}')
    if s.get('source_text_gold'):
        print(f'   gold: {s["source_text_gold"][:90]}')
    for name, _ in ASR_VARIANTS:
        print(f'   [{name}] asr: {asr_preds[name][s["idx"]][:90]}')
        print(f'   [{name}]  mt: {mt_preds[name][s["idx"]][:90]}')
    if s.get('reference_target'):
        print(f'   ref: {s["reference_target"][:90]}')